# AgentDiff on Colab

**Git diff for AI agents** — where two runs parted, why it happened, what it
cost, and what to change.

Three things happen in this notebook:

1. The full analysis pipeline runs on the bundled demo, with every report
   rendered **inline** — no downloads, no local server.
2. A **real open-weight model runs on Colab's GPU**, its per-token logprobs
   are captured, and two runs of it are compared against each other. This is
   the case AgentDiff is strongest on: a self-hosted model returns logprobs
   freely, so "did the model know it was going wrong?" is answerable from
   real numbers rather than estimates.
3. The reports are served as pages, for opening in their own tab.

Sections 1, 2 and 4 need no GPU. Section 3 does —
*Runtime → Change runtime type → T4 GPU*.

## 1 · Get the code

In [ ]:
!git clone --depth 1 --branch claude/deepcompare-ai-agents-9vyj2n \
    https://github.com/sugeerth/Agenttracing.git /content/agentdiff \
    2>/dev/null || (cd /content/agentdiff && git pull --ff-only)
%cd /content/agentdiff
!python -m unittest discover -s tests -q 2>&1 | tail -3

AgentDiff is pure standard-library Python — nothing to install, and the test
suite above is the fastest check that the clone is intact.

## 2 · The demo, end to end

Two agents, eight tasks, both instrumented. `generate.py` writes the
trajectories; `add_telemetry.py` adds the model-confidence variants.

In [ ]:
!python demo/generate.py > /dev/null && echo "flagship traces written"
!python demo/telemetry/add_telemetry.py > /dev/null && echo "telemetry variants written"
!ls demo/telemetry/traces | head -4

### Where the two agents parted

`batch` aligns every task's pair of trajectories, finds the first real
divergence, classifies it, and attributes what followed to it.

In [ ]:
!python -m deepcompare batch demo/telemetry/traces -o out --template web/app.html 2>&1 | tail -25

### Reading the reports inline

Every AgentDiff view is one self-contained HTML file, so Colab can render it
directly. `srcdoc` isolation keeps the report's CSS away from Colab's own.

In [ ]:
from IPython.display import HTML, display
from pathlib import Path


def show(path, height=900):
    """Render an AgentDiff report inside the notebook."""
    html = Path(path).read_text(encoding="utf-8")
    escaped = html.replace("&", "&amp;").replace('"', "&quot;")
    display(HTML(
        f'<iframe srcdoc="{escaped}" width="100%" height="{height}" '
        'style="border:1px solid #ddd;border-radius:8px"></iframe>'
    ))


show("out/report.html")

### The other views

Same data, four questions. Each is a different template over the same
reports, so switching view costs one flag.

In [ ]:
# One run in complete depth: waterfall, full step text, claims, confidence.
!python -m deepcompare batch demo/telemetry/traces -o out_explore \
    --template web/explore.html > /dev/null
show("out_explore/report.html", height=1000)

In [ ]:
# Sankey flows + attribute forest plot + Shapley credit allocation.
!python -m deepcompare batch demo/telemetry/traces -o out_flows \
    --template web/holistic.html > /dev/null
show("out_flows/report.html", height=1100)

In [ ]:
# 33 agents: which are interchangeable, which are redundant, which to use.
!python -m deepcompare select demo/fleet/traces -o out_select 2>&1 | tail -20
show("out_select/select.html", height=1000)

### The composable view

Same reports, arranged as a board of blocks: drag them between columns,
collapse what you don't need, remove what you never read. Ordering comes from
what the run has to say and what you actually open — and it is offered rather
than applied, since a layout you arranged by hand is better evidence than
anything inferred from your clicks.

Your arrangement follows a visitor id minted in your browser. Nothing is
transmitted; the **You** panel shows the id, everything held under it, and
erases the lot in one click. (Inside a Colab output frame the storage is
per-frame, so the arrangement lasts the session rather than following you
between notebooks.)

In [ ]:
!python web/build_blocks.py
!python -m deepcompare batch demo/telemetry/traces -o out_blocks \
    --template web/blocks.html > /dev/null
show("out_blocks/report.html", height=1100)

### Is the score telling the truth?

Outcome-only evaluation is blind by construction. These runs are built so
the *passing* one is the one that misbehaved: it hammered a failing lookup
three times, never recovered, and then wrote without ever having read the
booking. A leaderboard scores it identically to a clean pass.

The reverse case is in the same batch — a run that failed with a spotless
process, which is evidence about the grader rather than the agent.

None of this uses a judge or a re-run. It is counted from the trace.

In [ ]:
!python demo/process/generate.py
!python -m deepcompare batch demo/process/traces -o out_process 2>&1 | tail -12

In [ ]:
import json
from pathlib import Path

for path in sorted(Path("out_process").glob("report_*.json")):
    report = json.loads(path.read_text())
    for side in ("a", "b"):
        gap = report["process"][side]["gap"]
        flags = ", ".join(gap["raised"]) or "—"
        print(f'{report[side]["agent"]["name"]:<12} '
              f'{"PASS" if gap["success"] else "FAIL"}  '
              f'{gap["verdict"]:<24} {flags}')
    print()

In [ ]:
show("out_process/report.html", height=1000)

### Reliability: one lucky run is not a measurement

`pass^k` asks whether an agent works *every* time, not whether it ever
worked. An agent that solves a task 3 times out of 4 has a 0.75 success rate
and a **pass^4 of 0** — those describe very different things to anyone who
has to depend on it.

The advisory in this output is the honest part: at 3 runs per task, nothing
here supports ranking the two agents against each other.

In [ ]:
!python -m deepcompare runs demo/runs/traces -o out_runs 2>&1 | tail -22

### Analyses that need no second agent

A reference profile says what a normal run looks like; a cohort comparison
groups runs by model or version and reports each group's success rate with a
Wilson interval, so a difference is only called when the interval supports it.

In [ ]:
!python -m deepcompare profile demo/fleet/traces -o out_profile 2>&1 | head -12
print()
!python -m deepcompare cohort demo/fleet/traces --by model -o out_cohort 2>&1 | head -12

## 3 · A real open-weight model, with real logprobs

**This is the part Colab makes possible.** Everything above used the demo's
synthetic confidence, which is honestly labelled `synthetic-demo` in the
traces. A model running on this GPU returns genuine per-token logprobs, and
those are what the uncertainty analysis needs.

Set *Runtime → Change runtime type → **T4 GPU*** first. A small instruct
model is enough — swap `MODEL_ID` for any open-weight model that fits.

In [ ]:
!pip -q install "transformers>=4.44" torch accelerate 2>&1 | tail -1

In [ ]:
import json, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # any open-weight instruct model

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    print("no GPU detected — this runs on CPU, just slowly")
print("loaded", MODEL_ID, "on", model.device)


def generate_with_logprobs(prompt, max_new_tokens=120, temperature=0.2):
    """Generate text and return per-token logprobs in the OpenAI shape.

    top_logprobs is what lets AgentDiff compute entropy over the returned
    distribution instead of falling back to a floor.
    """
    text = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)

    started = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            top_p=0.95,
            return_dict_in_generate=True,
            output_scores=True,
        )
    elapsed = time.time() - started

    generated = out.sequences[0][inputs["input_ids"].shape[1]:]
    content = []
    for position, token_id in enumerate(generated):
        logprobs = torch.log_softmax(out.scores[position][0].float(), dim=-1)
        top = torch.topk(logprobs, k=5)
        content.append({
            "token": tok.decode(token_id),
            "logprob": float(logprobs[token_id]),
            "top_logprobs": [
                {"token": tok.decode(index), "logprob": float(value)}
                for value, index in zip(top.values, top.indices)
            ],
        })

    return {
        "text": tok.decode(generated, skip_special_tokens=True),
        "logprobs": {"content": content},
        "latency_s": round(elapsed, 4),
        "prompt_tokens": int(inputs["input_ids"].shape[1]),
        "eval_count": int(generated.shape[0]),
    }


probe = generate_with_logprobs("In one sentence: what is a Sankey diagram?")
print(probe["text"][:200])
print()
print(len(probe["logprobs"]["content"]), "tokens scored in",
      probe["latency_s"], "s")
print("weakest token:", min(probe["logprobs"]["content"],
                            key=lambda t: t["logprob"])["token"])

### Turning that into an AgentDiff trace

A tiny scripted agent — plan, then answer — gives a real trajectory to
analyse. It is recorded in the shape an Ollama-style runner emits, which the
adapter picks up automatically: token counts, wall-clock timing and logprobs
all carry through.

`dry_run` is the honest first move. It reports what the mapping *recovered*
— steps with text, timing, tokens, observations — before anything downstream
is allowed to depend on it.

In [ ]:
import sys
sys.path.insert(0, "/content/agentdiff")
from deepcompare.registry import convert, dry_run


def run_agent(task, expected, temperature=0.2):
    """Plan, then answer. Returns a payload in Ollama-runner shape."""
    plan = generate_with_logprobs(
        "Briefly state how you will answer this, in one sentence: " + task,
        max_new_tokens=60, temperature=temperature)
    answer = generate_with_logprobs(task, max_new_tokens=40,
                                    temperature=temperature)

    def turn(step):
        return {
            "model": MODEL_ID,
            "done": True,
            "eval_count": step["eval_count"],
            "prompt_eval_count": step["prompt_tokens"],
            "latency_s": step["latency_s"],
            "options": {"temperature": temperature},
            "message": {"role": "assistant", "content": step["text"]},
            "logprobs": step["logprobs"],
        }

    return {
        "meta": {
            "agent": {"name": f"local-t{temperature}", "model": MODEL_ID,
                      "version": "transformers"},
            "task": {"id": "t_capital", "prompt": task, "expected": expected},
            "success": expected.lower() in answer["text"].lower(),
            "answer": answer["text"].strip(),
        },
        "turns": [turn(plan), turn(answer)],
    }


TASK = "What is the capital of France? Answer with just the city name."
payload = run_agent(TASK, "Paris")

report = dry_run(payload)
print("format:", report["format"], "| detection confidence:", report["confidence"])
print("recovered:", report["fidelity"])
for note in report["notes"]:
    print("  note:", note)

trajectory = convert(payload)["trajectory"]
print("\nSteps, with telemetry the model actually produced:")
for step in trajectory["steps"]:
    telemetry = step.get("model")
    line = f"  {step['index']}  {step['type']:<8} {step['tokens']:>4} tok  {step['latency_s']:>6.2f}s"
    if telemetry:
        line += (f"  confidence={telemetry['confidence']:.3f}"
                 f"  weakest={telemetry['min_token_confidence']:.3f}"
                 f"  entropy={telemetry['entropy']:.3f}"
                 f"  ({telemetry['entropy_basis']})")
    print(line)

`entropy_basis` is the part worth reading. `top_k` means entropy was computed
over the distribution the model returned. `binary_floor` means only the
chosen token's probability was available, so the number is a floor rather
than an estimate — AgentDiff labels which one it had instead of presenting
both as the same measurement. `source` says `ollama-logprobs`, not
`synthetic-demo`.

### Comparing two real runs

One trace is a record; two are a diff. Running the same model at a second
temperature gives a genuine pair to compare — same task, same weights,
different sampling — and the full pipeline runs on it exactly as it did on
the demo.

In [ ]:
import os

os.makedirs("openweight_traces", exist_ok=True)


def save(payload):
    trajectory = convert(payload)["trajectory"]
    path = (f"openweight_traces/{trajectory['task']['id']}"
            f"__{trajectory['agent']['name']}.json")
    with open(path, "w") as handle:
        json.dump(trajectory, handle, indent=2)
    print("wrote", path,
          "| success:", trajectory["outcome"]["success"],
          "|", trajectory["totals"]["output_tokens"], "tokens")
    return path


save(payload)                                    # the temperature 0.2 run
save(run_agent(TASK, "Paris", temperature=1.2))  # a hotter second run

In [ ]:
!python -m deepcompare batch openweight_traces -o out_local \
    --template web/app.html 2>&1 | tail -20
show("out_local/report.html")

Two runs of one small model on one easy task will often agree, and when they
do AgentDiff says so rather than inventing a divergence. Point `run_agent` at
a harder task, or a second `MODEL_ID`, and the comparison has something to
find — the pipeline is identical either way.

## 4 · Serving the reports as pages

Inline rendering is usually enough. To open a report in its own tab, serve
the working directory and use Colab's port proxy.

In [ ]:
import subprocess

PORT = 8000
subprocess.Popen(
    ["python", "-m", "http.server", str(PORT)],
    cwd="/content/agentdiff",
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print(f"serving /content/agentdiff on :{PORT}")

try:
    from google.colab.output import eval_js
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    for page in ("out/report.html", "out_flows/report.html",
                 "out_select/select.html", "out_explore/report.html"):
        print("open:", url + page)
except Exception as exc:
    print("not in Colab, or the proxy is unavailable:", exc)

### Or download them

In [ ]:
from google.colab import files

files.download("out/report.html")

## Command reference

| question | command |
|---|---|
| Where did these two runs part, and what did it cost? | `deepcompare batch DIR -o out` |
| Which of N agents should I use? | `deepcompare select DIR -o out` |
| Is this failure real, or noise? | `deepcompare runs DIR -o out` (needs repeat runs) |
| Did my new version regress? | `deepcompare gate baseline/ candidate/` (exits non-zero) |
| Does this run follow our approved procedure? | `deepcompare check DIR --golden GOLDEN` |
| What does a normal run look like? | `deepcompare profile DIR -o out` |
| Does model family A beat B? | `deepcompare cohort DIR --by model` |
| I have traces in another format | `deepcompare convert IN.json --dry-run` |

Templates are interchangeable via `--template`: `web/app.html` (verdict
first), `web/explore.html` (one run in depth), `web/holistic.html` (Sankey
flows and credit allocation), `web/blocks.html` (the composable board),
`web/select.html` (fleet selection), `web/viewer.html` (the full D3
dashboard).

Bring your own traces by matching `SCHEMA.md`, or convert them — the registry
detects OpenTelemetry GenAI, OpenAI, Anthropic and Ollama shapes, and
`--dry-run` shows exactly what a conversion recovered before you trust it.